# Inference — Italian WITS

Generates summaries on Italian WITS test data using SigExt + Llama-3.1.

In [ ]:
!pip install -e /path/to/sm-sip

In [ ]:
from huggingface_hub import login
login()

## Configuration

In [ ]:
from sm_sip.config import SigExtConfig, InferenceConfig

sigext_config = SigExtConfig.from_preset("it", "10k-60t")
inference_config = InferenceConfig(
    lang="it",
    quantization="8bit",
    prompt_type="source_aware",
    num_test_samples=100,
)

## Load Data

In [ ]:
from sm_sip.data import get_test_data

test_data = get_test_data(
    lang="it",
    num_samples=inference_config.num_test_samples,
    skip_samples=sigext_config.skip_samples,
)

## Load Models

In [ ]:
from sm_sip.models import load_sigext_model, load_llm, create_summary_chain
from sm_sip.prompts import get_summary_prompt

# Load SigExt
sigext_model, sigext_tokenizer = load_sigext_model(sigext_config.model_id)

# Load LLM
llm_model, llm_tokenizer, gen_pipe = load_llm(
    inference_config.llm_model_id,
    inference_config.quantization,
)

# Create summary chain
prompt_template = get_summary_prompt("it", inference_config.prompt_type)
summary_chain = create_summary_chain(gen_pipe, prompt_template)

## Preprocess & Generate

In [ ]:
from sm_sip.models import preprocess_dataset
from sm_sip.pipelines import run_inference

# Extract salient sentences
processed_data = preprocess_dataset(test_data, sigext_model, sigext_tokenizer, lang="it")

# Generate summaries
results = run_inference(processed_data, summary_chain)

## Save Results

In [ ]:
from sm_sip.utils.io import save_results
from datetime import datetime

output = {
    "run_info": {
        "timestamp": datetime.now().isoformat(),
        "sigext_model": sigext_config.model_id,
        "llm_model": inference_config.llm_model_id,
        "quantization": inference_config.quantization,
        "prompt_type": inference_config.prompt_type,
        "num_samples": len(results),
    },
    "samples": results,
}
save_results(output, f"results/italian/inference_{inference_config.prompt_type}.json")

## Cleanup

In [ ]:
from sm_sip.utils.gpu import clear_gpu_memory
import gc

del llm_model, llm_tokenizer, gen_pipe, sigext_model
clear_gpu_memory()
print("Cleanup complete!")